In [ ]:
include( "../model.jl" )
include( "../prob.jl" )

using ThreadsX
using StatsBase
using Printf

using StatsPlots
using Distributions

# home, base = expfolder( "system" )
base = "/home/michaelnaps/prog/hpc-pass-data/system/"

In [ ]:
# System parameters.
N = 8

# Import noise, speed, and density lists.
include( "../0-params/nos-list.jl" )
include( "size-0-params.jl" )

# Color map.
# colormap = cgrad( [:black, :gray, :white, :mediumpurple], [0.0, 0.10, 0.40, 0.50, 0.75, 1.0] )
colormap = cgrad( :ice )

In [ ]:
# Initialize folder names.
folderdatalist = [findfolder( N, ρlist[i], slist[j], ηlist[k]; base=base ) for i ∈ 1:Nρ, j ∈ 1:Ns, k ∈ 1:Nη];

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*Ns, "Extracting critical noise information." )

# Initialize mean order and critical noise matrix.
μφdatalist = Array{defFloat,3}( undef, Nρ, Ns, Nη )

# Iterate through the data.
T = 100_000
@threads for i ∈ 1:Nρ; for j ∈ 1:Ns
    # Import order parameter data.
    φdata = [vec( readdlm( folderdatalist[i,j,k]*"order_$(T)-end.txt" ) ) for k ∈ 1:Nη]

    # Compute the mean order parameter.
    μφdatalist[i,j,:] = mean.( φdata )

    # Update progress meter.
    next!( meter )
end;  end

In [ ]:
# Extract the critical noise for each (ρ,s) pair.
ηcdata = [criticalnoise( ηlist, μφdatalist[i,j,:] ) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
j = 5

# Plot an example order-disorder trend.
plt = plot(; size=(400,250), legend=false )

for i ∈ 1:Nρ
    plot!( plt, ηlist, μφdatalist[i,j,:]; color=colormap[(i-1)/(Nρ-1)], marker=:circ, lw=2 )
end

plot!( plt; xlims=(0,ηlist[end]), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )

In [ ]:
i = 1

# Plot an example order-disorder trend.
plt = plot(; size=(400,250), legend=false )

for j ∈ 1:Ns
    plot!( plt, ηlist, μφdatalist[i,j,:]; color=colormap[(j-1)/(Ns-1)], marker=:circ, lw=2 )
end

plot!( plt; xlims=(0,ηlist[end]), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )

In [ ]:
# Build heatmap of the optimal JSD.
plt = plot(; size=(350,300) )
plot!( plt; right_margin=15pt, bottom_margin=0pt )

heatmap!( plt, ρlist, slist, ηcdata'; cmap=colormap, clims=(0,0.4) )

plot!( plt; xlims=10.0.^(ρmin,ρmax), xscale=:log10 )
plot!( plt; ylims=10.0.^(smin,smax), yscale=:log10 )

plot!( plt; xlabel="density, "*L"ρ", ylabel="speed, "*L"s" )

In [ ]:
if false
    # Save data to pass to the HPC.
    for i ∈ 1:Nρ, j ∈ 1:Ns
        # folder = folderdatalist[i,j,1]*"../"
        folder = createfolder( replace( folderdatalist[i,j,1]*"../", home => "../../hpc-pass-data/" ) )
        writedlm( folder*"order-0.5.txt", ηcdata[i,j] )
    end
end